# Locus Coeruleus & Peri-LC Neuromodulator Receptor Expression (10x snRNA-seq)

This notebook examines neuromodulator receptor expression in **locus coeruleus (LC)**
and surrounding pontine cell types using 10x snRNA-seq from the Allen Brain Cell Atlas.

### Dissection Region
- **P** — pons dissection, covering LC, parabrachial, laterodorsal tegmental,
  and other pontine nuclei

### Key cell type
- **NTS Dbh Glut** — noradrenergic neurons (Dbh+), the core LC cell type

### Gene Panel (28 receptors — full 10x panel)
- **Serotonin (14)**: Htr1a, Htr1b, Htr1d, Htr1f, Htr2a, Htr2b, Htr2c, Htr3a, Htr3b, Htr4, Htr5a, Htr5b, Htr6, Htr7
- **Norepinephrine (9)**: Adra1a, Adra1b, Adra1d, Adra2a, Adra2b, Adra2c, Adrb1, Adrb2, Adrb3
- **Dopamine (5)**: Drd1, Drd2, Drd3, Drd4, Drd5

### Important Caveat
The 10x `P` dissection covers all pontine structures together. Individual nuclei
(LC, SLC, PB, LDT, etc.) cannot be isolated from 10x data alone — see the MERFISH
notebook for structure-specific analysis.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

## 1. Initialize Cache and Load Metadata

In [ ]:
download_base = Path('../../data/abc_atlas')
abc_cache = AbcProjectCache.from_s3_cache(download_base)
print(f'Manifest: {abc_cache.current_manifest}')

In [ ]:
cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print(f'Total cells in WMB-10X: {len(cell):,}')

In [ ]:
gene = abc_cache.get_metadata_dataframe(directory='WMB-10X', file_name='gene')
gene.set_index('gene_identifier', inplace=True)
print(f'Total genes: {len(gene):,}')

In [ ]:
cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
)
cluster_details.set_index('cluster_alias', inplace=True)
cell_extended = cell.join(cluster_details, on='cluster_alias')
print(f'Cells with annotations: {len(cell_extended):,}')

## 2. Identify Pontine Cells

Select cells from the pons (P) dissection region, which contains the locus coeruleus
and surrounding pontine structures.

In [ ]:
pons_cells = cell_extended[cell_extended['region_of_interest_acronym'] == 'P']
print(f'Cells in P (pons) region: {len(pons_cells):,}')

print(f'\nBy neurotransmitter:')
for nt, count in pons_cells.groupby('neurotransmitter', observed=True).size().sort_values(ascending=False).items():
    print(f'  {nt}: {count:,}')

print(f'\nBy class:')
for cls, count in pons_cells.groupby('class', observed=True).size().sort_values(ascending=False).items():
    print(f'  {cls}: {count:,}')

print(f'\nAll subclasses in P (>= 50 cells):')
sc_counts = pons_cells.groupby('subclass', observed=True).size().sort_values(ascending=False)
for sc_name, count in sc_counts.items():
    if count >= 50:
        nt_vals = pons_cells[pons_cells['subclass'] == sc_name]['neurotransmitter'].value_counts()
        nt = nt_vals.index[0] if len(nt_vals) > 0 else '?'
        print(f'  {sc_name}: {count:,} cells  [{nt}]')

In [ ]:
# Select subclasses with >= 50 cells
min_cells_select = 50
selected_subclasses = sc_counts[sc_counts >= min_cells_select].index.tolist()

pons_selected = pons_cells[pons_cells['subclass'].isin(selected_subclasses)].copy()
print(f'Selected: {len(selected_subclasses)} subclasses, {len(pons_selected):,} cells')

# Highlight Dbh+ (noradrenergic) neurons
dbh_mask = pons_selected['subclass'].str.contains('Dbh', na=False)
print(f'\nDbh+ (noradrenergic) neurons: {dbh_mask.sum():,}')

## 3. Define Receptor Gene Lists

In [ ]:
serotonin_receptors = [
    'Htr1a', 'Htr1b', 'Htr1d', 'Htr1f',
    'Htr2a', 'Htr2b', 'Htr2c',
    'Htr3a', 'Htr3b',
    'Htr4', 'Htr5a', 'Htr5b', 'Htr6', 'Htr7'
]
norepinephrine_receptors = [
    'Adra1a', 'Adra1b', 'Adra1d',
    'Adra2a', 'Adra2b', 'Adra2c',
    'Adrb1', 'Adrb2', 'Adrb3'
]
dopamine_receptors = ['Drd1', 'Drd2', 'Drd3', 'Drd4', 'Drd5']

all_receptors = serotonin_receptors + norepinephrine_receptors + dopamine_receptors
available_genes = gene[gene['gene_symbol'].isin(all_receptors)]
receptor_genes = [g for g in all_receptors if g in set(available_genes['gene_symbol'])]
gene_ensembl_ids = available_genes.index.tolist()

print(f'Found {len(receptor_genes)}/{len(all_receptors)} receptor genes')

## 4. Load Expression Data

In [ ]:
pons_matrices = pons_selected.groupby('feature_matrix_label').size()
print('Expression matrices containing P cells:')
for mat, count in pons_matrices.items():
    print(f'  {mat}: {count:,} cells')

In [ ]:
neuronal_csv = 'lc_10x_expression.csv'
meta_csv = 'lc_10x_metadata.csv'

if os.path.exists(neuronal_csv):
    expression_data = pd.read_csv(neuronal_csv, index_col=0)
    print(f'Loaded from {neuronal_csv}: {expression_data.shape[0]:,} cells x {expression_data.shape[1]} genes')
else:
    expression_frames = []
    for matrix_label in pons_matrices.index:
        dataset_label = pons_selected[
            pons_selected['feature_matrix_label'] == matrix_label
        ]['dataset_label'].iloc[0]
        file_name = f'{matrix_label}/log2'

        print(f'\nLoading {file_name} from {dataset_label}...')
        file_path = abc_cache.get_file_path(directory=dataset_label, file_name=file_name)

        adata = anndata.read_h5ad(file_path, backed='r')

        gene_mask = adata.var.index.isin(gene_ensembl_ids)
        gene_filtered = adata.var[gene_mask]

        cell_labels = pons_selected[
            pons_selected['feature_matrix_label'] == matrix_label
        ].index
        cell_mask = adata.obs.index.isin(cell_labels)

        print(f'  Cells in matrix: {len(adata.obs):,}')
        print(f'  Pons cells found: {cell_mask.sum():,}')
        print(f'  Receptor genes found: {gene_mask.sum()}')

        cell_idx = np.where(cell_mask)[0]
        gene_idx = np.where(gene_mask)[0]
        subset = adata[cell_idx, gene_idx].to_memory()

        expr_df = subset.to_df()
        expr_df.columns = gene_filtered['gene_symbol'].values
        expression_frames.append(expr_df)

        adata.file.close()
        del adata

    expression_data = pd.concat(expression_frames)
    expression_data = expression_data[receptor_genes]
    expression_data.to_csv(neuronal_csv)
    print(f'\nSaved to {neuronal_csv}')

# Save metadata
pons_selected.loc[expression_data.index,
    ['subclass', 'supertype', 'class', 'neurotransmitter']
].to_csv(meta_csv)

print(f'Total: {expression_data.shape[0]:,} cells x {expression_data.shape[1]} genes')
print(f'Saved metadata to {meta_csv}')

## 5. Build AnnData and Prepare for Dot Plot

In [ ]:
expression_data = expression_data[receptor_genes]

meta = pons_selected.loc[expression_data.index].copy()
meta['subclass_short'] = meta['subclass'].apply(
    lambda x: re.sub(r'^\d+\s+', '', str(x))
)
meta['supertype_short'] = meta['supertype'].apply(
    lambda x: re.sub(r'^\d+\s+', '', str(x))
)
meta['class_short'] = meta['class'].apply(
    lambda x: re.sub(r'^\d+\s+', '', str(x))
)

adata_pons = anndata.AnnData(
    X=expression_data.values,
    obs=meta[['subclass', 'supertype', 'class', 'neurotransmitter',
              'subclass_short', 'supertype_short', 'class_short']].copy(),
    var=pd.DataFrame(index=receptor_genes)
)

adata_pons.obs['subclass_short'] = pd.Categorical(adata_pons.obs['subclass_short'])
adata_pons.obs['supertype_short'] = pd.Categorical(adata_pons.obs['supertype_short'])

n_subclasses = adata_pons.obs['subclass_short'].cat.categories.size
n_supertypes = adata_pons.obs['supertype_short'].cat.categories.size
print(adata_pons)
print(f'\n{n_subclasses} subclasses, {n_supertypes} supertypes')

# Subclass overview
print(f'\nSubclasses (>= 50 cells):')
for sc_name, cnt in meta.groupby('subclass_short', observed=True).size().sort_values(ascending=False).items():
    if cnt >= 50:
        nt = meta[meta['subclass_short'] == sc_name]['neurotransmitter'].mode()
        nt_str = nt.iloc[0] if len(nt) > 0 else ''
        print(f'  {sc_name}: {cnt:,} ({nt_str})')

## 6. Dot Plot: Receptor Expression by Subclass

All cell types in the pontine dissection with >= 50 cells.

In [ ]:
receptor_groups = {}
sero_available = [g for g in serotonin_receptors if g in receptor_genes]
ne_available = [g for g in norepinephrine_receptors if g in receptor_genes]
da_available = [g for g in dopamine_receptors if g in receptor_genes]
if sero_available: receptor_groups['Serotonin (5-HT)'] = sero_available
if ne_available: receptor_groups['Norepinephrine (NE)'] = ne_available
if da_available: receptor_groups['Dopamine (DA)'] = da_available

# Sort subclasses: Dbh+ first, then by neurotransmitter, then alpha
all_sc = adata_pons.obs['subclass_short'].cat.categories.tolist()

def sc_sort_key(name):
    if 'Dbh' in name:
        return (0, name)
    elif 'Tph2' in name or 'Sero' in name:
        return (1, name)
    elif 'Glut' in name:
        return (2, name)
    elif 'Gaba' in name or 'Gly' in name:
        return (3, name)
    elif 'Chol' in name:
        return (4, name)
    else:
        return (5, name)

sorted_sc = sorted(all_sc, key=sc_sort_key)
adata_pons.obs['subclass_short'] = pd.Categorical(
    adata_pons.obs['subclass_short'], categories=sorted_sc, ordered=True
)

n_sc = len(sorted_sc)
print(f'Subclasses: {n_sc}')

dp = sc.pl.dotplot(
    adata_pons,
    var_names=receptor_groups,
    groupby='subclass_short',
    standard_scale='var',
    cmap='Reds',
    figsize=(16, max(6, n_sc * 0.4)),
    show=False,
    return_fig=True
)
dp.style(dot_edge_color='black', dot_edge_lw=0.5)
dp.savefig('dotplot_LC_receptors_by_subclass.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dotplot_LC_receptors_by_subclass.png')

## 7. Dot Plot: Neuronal Subclasses Only

Focused view on neurons, excluding glia and vascular cells.

In [ ]:
# Filter to neuronal subclasses
non_neuronal_keywords = ['Astro', 'OPC', 'Oligo', 'Microglia', 'Endo',
                         'Peri', 'SMC', 'VLMC', 'Ependymal', 'BAM', 'CHOR']
neuronal_sc = [s for s in sorted_sc
               if not any(kw in s for kw in non_neuronal_keywords)]
print(f'Neuronal subclasses: {len(neuronal_sc)}')

adata_neuron = adata_pons[adata_pons.obs['subclass_short'].isin(neuronal_sc)].copy()
adata_neuron.obs['subclass_short'] = pd.Categorical(
    adata_neuron.obs['subclass_short'],
    categories=[s for s in sorted_sc if s in neuronal_sc],
    ordered=True
)

n_neuron = len(neuronal_sc)
dp2 = sc.pl.dotplot(
    adata_neuron,
    var_names=receptor_groups,
    groupby='subclass_short',
    standard_scale='var',
    cmap='Reds',
    figsize=(16, max(6, n_neuron * 0.4)),
    show=False,
    return_fig=True
)
dp2.style(dot_edge_color='black', dot_edge_lw=0.5)
dp2.savefig('dotplot_LC_receptors_neurons.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dotplot_LC_receptors_neurons.png')

## 8. Finer View: Grouped by Supertype

In [ ]:
min_cells = 50
supertype_counts = adata_neuron.obs.groupby('supertype_short', observed=True).size()
valid_supertypes = supertype_counts[supertype_counts >= min_cells].index.tolist()

adata_st = adata_neuron[adata_neuron.obs['supertype_short'].isin(valid_supertypes)].copy()
adata_st.obs['supertype_short'] = pd.Categorical(adata_st.obs['supertype_short'])

n_st = len(valid_supertypes)
print(f'Supertypes with >= {min_cells} cells: {n_st}')

dp3 = sc.pl.dotplot(
    adata_st,
    var_names=receptor_groups,
    groupby='supertype_short',
    standard_scale='var',
    cmap='Reds',
    figsize=(16, max(8, n_st * 0.35)),
    show=False,
    return_fig=True
)
dp3.style(dot_edge_color='black', dot_edge_lw=0.5)
dp3.savefig('dotplot_LC_receptors_by_supertype.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dotplot_LC_receptors_by_supertype.png')

## 9. Focus: Noradrenergic (Dbh+) Neurons — Receptor Profile

Detailed look at the receptor expression profile of Dbh+ noradrenergic
neurons (NTS Dbh Glut), the primary output cells of the locus coeruleus.
With 10x we get the full 28-receptor panel vs only 11 in MERFISH.

In [ ]:
# Identify Dbh+ neurons
dbh_mask = meta['subclass_short'].str.contains('Dbh', na=False)
dbh_meta = meta[dbh_mask].copy()
dbh_expr = expression_data.loc[dbh_meta.index]

print(f'Dbh+ (noradrenergic) neurons: {len(dbh_meta):,}')
print(f'\nBy supertype:')
for st, cnt in dbh_meta.groupby('supertype_short', observed=True).size().sort_values(ascending=False).items():
    print(f'  {st}: {cnt:,}')

# Fraction expressing and mean expression
dbh_frac = (dbh_expr > 0).mean()
dbh_mean = dbh_expr.mean()

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

x = np.arange(len(receptor_genes))

# Color by receptor family
colors = []
for g in receptor_genes:
    if g.startswith('Htr'): colors.append('#e41a1c')
    elif g.startswith('Adr'): colors.append('#377eb8')
    else: colors.append('#4daf4a')

ax = axes[0]
ax.bar(x, dbh_frac[receptor_genes], color=colors, alpha=0.8)
ax.set_ylabel('Fraction expressing')
ax.set_title(f'Receptor Expression in Dbh+ (NE) Neurons (n={len(dbh_meta):,})',
             fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(receptor_genes, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1)

ax = axes[1]
ax.bar(x, dbh_mean[receptor_genes], color=colors, alpha=0.8)
ax.set_ylabel('Mean log2 expression')
ax.set_title(f'Mean Receptor Expression in Dbh+ (NE) Neurons',
             fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(receptor_genes, rotation=45, ha='right', fontsize=8)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e41a1c', alpha=0.8, label='Serotonin'),
    Patch(facecolor='#377eb8', alpha=0.8, label='Norepinephrine'),
    Patch(facecolor='#4daf4a', alpha=0.8, label='Dopamine'),
]
axes[0].legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('barplot_LC_Dbh_receptor_profile_10x.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: barplot_LC_Dbh_receptor_profile_10x.png')

In [ ]:
# Receptor profile table
print('=' * 60)
print('Receptor Profile of Dbh+ (Noradrenergic) Neurons (10x)')
print('=' * 60)
print(f'\n{"Gene":>10s}  {"Frac":>10s}  {"Mean":>10s}')
print('-' * 35)
for g in receptor_genes:
    print(f'{g:>10s}  {dbh_frac[g]:>10.3f}  {dbh_mean[g]:>10.3f}')

## 10. Heatmap: Mean Expression and Fraction Expressing by Subclass

In [ ]:
# Mean expression and fraction by subclass (neuronal only)
neuron_meta = meta[meta['subclass_short'].isin(neuronal_sc)].copy()
neuron_expr = expression_data.loc[neuron_meta.index]

expr_df = pd.DataFrame(neuron_expr.values, index=neuron_meta.index, columns=receptor_genes)
expr_df['subclass'] = neuron_meta['subclass_short'].values

# Sort order from dot plot
sc_order = [s for s in sorted_sc if s in neuronal_sc]

sc_mean = expr_df.groupby('subclass', observed=True)[receptor_genes].mean().loc[sc_order]
sc_frac = expr_df.groupby('subclass', observed=True)[receptor_genes].apply(
    lambda x: (x > 0).mean()
).loc[sc_order]

n_sc_neuron = len(sc_order)

fig, axes = plt.subplots(1, 2, figsize=(22, max(5, n_sc_neuron * 0.4)))

sns.heatmap(sc_mean, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Mean log2 expr'},
            annot_kws={'size': 6})
axes[0].set_title('Mean Expression by Subclass (Neurons)', fontweight='bold')
axes[0].set_ylabel('')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)

sns.heatmap(sc_frac, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': 'Fraction expressing'},
            annot_kws={'size': 6})
axes[1].set_title('Fraction Expressing by Subclass (Neurons)', fontweight='bold')
axes[1].set_ylabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('heatmap_LC_receptors_by_subclass_10x.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: heatmap_LC_receptors_by_subclass_10x.png')

## 11. Enrichment: Log2 Fold-Change vs Overall Pontine Mean

In [ ]:
overall_mean = expr_df[receptor_genes].mean()
pseudocount = 0.01
log2fc = np.log2((sc_mean + pseudocount) / (overall_mean + pseudocount))

fig, ax = plt.subplots(figsize=(18, max(5, n_sc_neuron * 0.45)))
sns.heatmap(log2fc, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.5, vmin=-3, vmax=3,
            cbar_kws={'label': 'log2 FC vs pontine mean'},
            annot_kws={'size': 6})
ax.set_title('Cell-Type-Specific Receptor Enrichment in Pons (10x)',
             fontweight='bold', fontsize=13)
ax.set_ylabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('heatmap_LC_receptors_enrichment_10x.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: heatmap_LC_receptors_enrichment_10x.png')

print('\nKey enrichments for Dbh+ neurons (|log2FC| > 0.5):')
dbh_sc = [s for s in sc_order if 'Dbh' in s]
for sc_name in dbh_sc:
    hits = []
    for g in receptor_genes:
        fc = log2fc.loc[sc_name, g]
        if abs(fc) > 0.5:
            direction = '+' if fc > 0 else '-'
            hits.append(f'{g}({direction}{abs(fc):.1f})')
    if hits:
        print(f'  {sc_name}: {{", ".join(hits)}}')

## 12. Summary Statistics

In [ ]:
print('=' * 80)
print('Mean Expression (log2) by Subclass — Pons Neurons (10x snRNA-seq)')
print('=' * 80)
display(sc_mean.round(3))

print('\n' + '=' * 80)
print('Fraction Expressing by Subclass — Pons Neurons (10x snRNA-seq)')
print('=' * 80)
display(sc_frac.round(3))

# Save CSVs
sc_mean.to_csv('lc_10x_subclass_mean_expression.csv')
sc_frac.to_csv('lc_10x_subclass_frac_expressing.csv')
log2fc.to_csv('lc_10x_subclass_enrichment.csv')
print('\nSaved summary CSVs.')

print(f'\n{"=" * 80}')
print('Summary')
print(f'{"=" * 80}')
print(f'Total cells: {len(expression_data):,}')
print(f'Neuronal subclasses: {n_sc_neuron}')
print(f'Dbh+ (NE) neurons: {dbh_mask.sum():,}')
print(f'Receptor genes: {len(receptor_genes)}')
print(f'\nAdvantage over MERFISH: {len(receptor_genes)} receptors vs 11 in MERFISH panel')